In [ ]:
import os
import time
from pathlib import Path
import jax
import mujoco
from mujoco import mjx
import viser
from mjviser import ViserMujocoScene

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [ ]:
path = Path.cwd() / "1d_pusher.xml"
model = mujoco.MjModel.from_xml_path(str(path))
mjx_model = mjx.put_model(model, contact_approximation="log barrier")

server = viser.ViserServer()
server.scene.add_grid("grid")
scene = ViserMujocoScene(server, model, num_envs=1)
scene.camera_tracking_enabled = False

In [ ]:
mjx_data = mjx.make_data(mjx_model)
scene.update_from_mjdata(mjx_data)

step = lambda mjx_data: mjx.step(mjx_model, mjx_data)
step = jax.jit(step)

while True:
    ctrl = mjx_data.ctrl.at[0].set(0.2 * mjx_data.time)
    mjx_data = mjx_data.replace(ctrl=ctrl)

    tic = time.time()
    mjx_data = step(mjx_data)
    print(f"\rStep took {(time.time() - tic) * 1e3:.3f} ms", end=" " * 10)

    scene.update_from_mjdata(mjx_data)

    remain = model.opt.timestep - (time.time() - tic)
    if remain > 0:
        time.sleep(remain)